In [26]:
import matplotlib.pyplot as plt
import kagglehub
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [27]:

# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\margi\.cache\kagglehub\datasets\subhaditya\fer2013plus\versions\1


In [28]:
# Set up data transformations for training and testing
data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomCrop(48, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
}

# Paths to dataset
train_dir = path + "/fer2013plus/fer2013/train"
test_dir = path + "/fer2013plus/fer2013/test"

# Load datasets using ImageFolder
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms["test"])

print("Train and Test Datasets Loaded!")


Train and Test Datasets Loaded!


In [29]:
import numpy as np

# Get the frequency of each class
class_counts = np.bincount(train_dataset.targets)

# Calculate class weights inversely proportional to the frequency of each class
# Calculate class weights inversely proportional to class frequencies
# Adjust class weights to smooth the impact of underrepresented classes

class_weights = 1. / np.sqrt(class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)


# Print class weights for verification
print("Class weights:", class_weights)


Class weights: tensor([0.0201, 0.0778, 0.0724, 0.0392, 0.0115, 0.0098, 0.0169, 0.0168])


In [30]:
from torch.utils.data import WeightedRandomSampler

# Create sample weights for each data point in the training dataset
sample_weights = [class_weights[label] for label in train_dataset.targets]

# Create WeightedRandomSampler
# better with replacement 
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# Create DataLoader using the sampler for stratified sampling
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("DataLoader with Stratified Sampling created!")

DataLoader with Stratified Sampling created!


In [31]:
# Define the CNN architecture
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(in_features=128 * 6 * 6, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=8)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)  # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Instantiate the model

print("CNN Model Defined!")

CNN Model Defined!


In [32]:
import torch
import torch.nn as nn
import torch.optim as optim

# Check if CUDA is available and set the device to GPU if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move your model to the GPU (if available)
model = CNN().to(device)  # This assumes you have defined the CNN model

In [33]:
# Define the loss function (CrossEntropyLoss) and optimizer (Adam)
criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)  # Move loss function to GPU
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [34]:
# Training loop
num_epochs = 3
for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU

        optimizer.zero_grad()  # Zero the gradients

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Print loss every 100 batches
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{len(train_loader)}], Loss: {running_loss / 100:.4f}")
            running_loss = 0.0


Epoch [1/3], Step [100/888], Loss: 2.0758
Epoch [1/3], Step [200/888], Loss: 2.0489
Epoch [1/3], Step [300/888], Loss: 1.9974
Epoch [1/3], Step [400/888], Loss: 1.9602
Epoch [1/3], Step [500/888], Loss: 1.9180
Epoch [1/3], Step [600/888], Loss: 1.9037
Epoch [1/3], Step [700/888], Loss: 1.8286
Epoch [1/3], Step [800/888], Loss: 1.8106
Epoch [2/3], Step [100/888], Loss: 1.7404
Epoch [2/3], Step [200/888], Loss: 1.6803
Epoch [2/3], Step [300/888], Loss: 1.6421
Epoch [2/3], Step [400/888], Loss: 1.6182
Epoch [2/3], Step [500/888], Loss: 1.5999
Epoch [2/3], Step [600/888], Loss: 1.5607
Epoch [2/3], Step [700/888], Loss: 1.5116
Epoch [2/3], Step [800/888], Loss: 1.5498
Epoch [3/3], Step [100/888], Loss: 1.4832
Epoch [3/3], Step [200/888], Loss: 1.4258
Epoch [3/3], Step [300/888], Loss: 1.4251
Epoch [3/3], Step [400/888], Loss: 1.3641
Epoch [3/3], Step [500/888], Loss: 1.4108
Epoch [3/3], Step [600/888], Loss: 1.3289
Epoch [3/3], Step [700/888], Loss: 1.3729
Epoch [3/3], Step [800/888], Loss:

In [35]:
from sklearn.metrics import f1_score, classification_report

# Ensure the model is in evaluation mode
model.eval()

# Initialize counters and storage for predictions and labels
correct = 0
total = 0
all_preds = []  # To store all predictions
all_labels = []  # To store all true labels

# Set device (either 'cuda' or 'cpu')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Ensure the model is on the correct device

with torch.no_grad():
    for inputs, labels in test_loader:
        # Move data to the appropriate device
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, predicted = torch.max(outputs, 1)

        # Calculate number of correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Store predictions and true labels for later metrics calculation
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate accuracy
accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

# Calculate F1 score (weighted or other averaging methods)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(f"F1 Score (Weighted): {f1:.4f}")

# Generate classification report
# Ensure 'train_dataset.classes' gives class names in the correct format (a list)
class_names = train_dataset.classes  # This is a list of class names, e.g. ['anger', 'fear', 'happiness', ...]

report = classification_report(all_labels, all_preds, target_names=class_names)
print(report)



Test Accuracy: 0.5223
F1 Score (Weighted): 0.5624
              precision    recall  f1-score   support

       anger       0.49      0.38      0.42       644
    contempt       0.05      0.33      0.08        51
     disgust       0.06      0.65      0.11        57
        fear       0.12      0.40      0.19       167
   happiness       0.79      0.61      0.69      1827
     neutral       0.72      0.48      0.58      2597
     sadness       0.34      0.35      0.34       856
    surprise       0.65      0.75      0.70       900

    accuracy                           0.52      7099
   macro avg       0.40      0.49      0.39      7099
weighted avg       0.64      0.52      0.56      7099



In [36]:
# Save the trained model to a file
torch.save(model.state_dict(), 'cnn_model_strat.pth')
print("Model Saved as cnn_model.pth")

Model Saved as cnn_model.pth


In [37]:
# Prepare to count predictions for each class
classes = train_dataset.classes  # Get class names from the dataset
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# Disable gradients for evaluation
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)  # Ensure to move data to the correct device
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)

        # Collect correct predictions for each class
        for label, prediction in zip(labels.cpu(), predictions.cpu()):
            label_name = classes[label.item()]  # Use .item() to convert label to integer index
            prediction_name = classes[prediction.item()]  # Use .item() to convert prediction to integer index

            if label_name == prediction_name:  # Compare class names
                correct_pred[label_name] += 1
            total_pred[label_name] += 1

# Print accuracy for each class
for classname, correct_count in correct_pred.items():
    if total_pred[classname] > 0:
        accuracy = 100 * float(correct_count) / total_pred[classname]
        print(f'Accuracy for class: {classname:10s} is {accuracy:.1f} %')
    else:
        print(f'No predictions for class: {classname}')


Accuracy for class: anger      is 37.6 %
Accuracy for class: contempt   is 33.3 %
Accuracy for class: disgust    is 64.9 %
Accuracy for class: fear       is 40.1 %
Accuracy for class: happiness  is 61.1 %
Accuracy for class: neutral    is 48.3 %
Accuracy for class: sadness    is 35.4 %
Accuracy for class: surprise   is 74.7 %
